# META-CXR Training on Kaggle (2x T4 GPU)

This notebook trains the unified META-CXR model on the MIMIC-CXR-JPG dataset using 2x T4 GPUs via PyTorch DistributedDataParallel. The training hyperparameters are aligned with the GCP L4 VM encoder-comparison setup: max epoch 15, LR 1e-4, effective batch 16, seed 42, and GCS checkpoint resume/upload.

**Prerequisites:**
- Kaggle accelerator set to **GPU T4 x2**
- Internet access enabled
- Three data datasets attached as Kaggle input:
  - **`mimic-cxr-jpg-lite`** - JPG images + metadata CSVs
  - **`mimic-cxr-p10-processed`** - preprocessed train/val/test CSVs
  - **`mimic-cxr-reported`** - `mimic_cxr_cleaned.csv` + report text files
- Kaggle Secret **`WANDB_API_KEY`** for W&B
- Kaggle Secret **`GCS_SERVICE_ACCOUNT`** for writing to `gs://meta-cxr-checkpoint`

**Steps:** Run Cell 0 first, then cells 1->6 in order. Configure encoder toggles in Cell 6 before launching training. Cell 6 resumes from GCS and uploads output to GCS after the subprocess exits. Cell 8 can be run manually to upload the latest local output again.


## Cell 0 - Load Kaggle Secrets

Loads `GCS_SERVICE_ACCOUNT` and `WANDB_API_KEY` from Kaggle Secrets before dependency setup/training.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GCS_SERVICE_ACCOUNT")
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")

os.environ["GCS_SERVICE_ACCOUNT"] = secret_value_0
os.environ["WANDB_API_KEY"] = secret_value_1

print("Kaggle secrets loaded: GCS_SERVICE_ACCOUNT, WANDB_API_KEY")


## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Kaggle has PyTorch, torchvision, numpy, pandas, scikit-learn preinstalled.
# Install only the packages that are missing.
packages = [
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "scikit-image",           # latest stable; io/transform APIs are unchanged
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal",       # provides health_multimodal used by biovil_t
    "timm>=0.9.0",            # required when the optional Swin encoder is enabled
    "spacy",                  # latest 3.x; stable spacy.load() API
    "nltk>=3.9",              # >=3.9 required by textblob pre-installed on Kaggle
    "google-cloud-storage",
    "transformers==4.44.2",   # pin for Qformer.py compatibility (apply_chunking_to_forward et al.)
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True,
)

# Install peft at the specific commit used by the project
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08"],
    check=True,
)

# Download NLTK data required by the METEOR scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Download spacy English model required by blip2.py (spacy.load("en_core_web_sm"))
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

# Detect Java installation (needed for METEOR/ROUGE scoring)
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True,
)
JAVA_HOME_DETECTED = result.stdout.strip()
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

# Verify GPU count
import torch
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


## Cell 2 — Weights & Biases Setup

Đăng nhập wandb bằng Kaggle Secret `WANDB_API_KEY`.

**Cách thêm secret trên Kaggle:**  
Notebook Settings → Add-ons → Secrets → **Name:** `WANDB_API_KEY` → **Value:** API key của bạn.

In [ ]:
import os

if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        print("wandb: API key loaded from Kaggle Secrets")
    except Exception as e:
        print(f"wandb: Could not load from Kaggle Secrets ({e}) — using pre-configured key if available")
else:
    print("wandb: API key loaded from Cell 0")

import wandb
wandb.login()
print("wandb: Logged in successfully")


## Cell 3 — Clone GitHub Repository

In [17]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest changes...")
    !git -C {REPO_DIR} pull

# Change working directory to repo root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

Cloning into '/kaggle/working/META-CXR'...
remote: Enumerating objects: 444, done.
remote: Counting objects: 100% (444/444), done.
remote: Compressing objects: 100% (266/266), done.
remote: Total 444 (delta 202), reused 409 (delta 167), pack-reused 0 (from 0)
Receiving objects: 100% (444/444), 26.69 MiB | 35.31 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Working directory: /kaggle/working/META-CXR
total 192
drwxr-xr-x 12 root root  4096 May 11 07:44 .
drwxr-xr-x  4 root root  4096 May 11 07:44 ..
drwxr-xr-x  2 root root  4096 May 11 07:44 assets
drwxr-xr-x  3 root root  4096 May 11 07:44 biovil_t
-rw-r--r--  1 root root   538 May 11 07:44 build_container.sh
drwxr-xr-x  3 root root  4096 May 11 07:44 checkpoints
-rw-r--r--  1 root root  8919 May 11 07:44 CHECKPOINT_WORKFLOW.md
drwxr-xr-x  2 root root  4096 May 11 07:44 configs
-rw-r--r--  1 root root    90 May 11 07:44 Dockerfile
-rw-r--r--  1 root root  7234 May 11 07:44 generate_mimic_cxr_cleaned.ipynb
drwxr-xr-x  8 root root

## Cell 4 — Verify Kaggle Input Datasets & Load Reports CSV

Two datasets must be attached to this notebook:

| Dataset | Slug | Contents |
|---------|------|----------|
| MIMIC-CXR-JPG-LITE | `mimic-cxr-jpg-lite` | JPG images at `p10/…` + metadata CSVs |
| mimic-cxr-reported | `mimic-cxr-reported` | `mimic_cxr_cleaned.csv` (pre-built) + `.txt` reports |

`mimic_cxr_cleaned.csv` đã được tạo sẵn trong dataset `mimic-cxr-reported`. Cell này chỉ verify paths và load file CSV, không cần rebuild mỗi session.

In [11]:
import os
import shutil
import time
import yaml

# ── Load Kaggle dataset config ────────────────────────────────────────────────
with open("configs/kaggle_datasets.yaml") as f:
    CFG = yaml.safe_load(f)

IMAGES_SLUG   = CFG["datasets"]["images"]["slug"]
REPORTS_SLUG  = CFG["datasets"]["reports"]["slug"]
REQUIRED_CSVS = CFG["datasets"]["images"]["required_files"]
CLEANED_CSV   = CFG["datasets"]["reports"]["cleaned_csv_filename"]
REPORTS_LOCAL = CFG["working"]["cleaned_csv_path"]
PROCESSED_SLUG = CFG["datasets"]["processed"]["slug"]
PROCESSED_REQUIRED = CFG["datasets"]["processed"]["required_files"]

def elapsed(start):
    return f"{time.perf_counter() - start:.2f}s"

def find_mount(slug, override_env=None):
    override = os.environ.get(override_env) if override_env else None
    if override and os.path.isdir(override):
        return override

    for root in CFG["mount_search_roots"]:
        direct = os.path.join(root, slug)
        if os.path.isdir(direct):
            return direct

        if not os.path.isdir(root):
            continue

        try:
            for entry in os.scandir(root):
                if not entry.is_dir():
                    continue
                if entry.name == slug:
                    return entry.path
                nested = os.path.join(entry.path, slug)
                if os.path.isdir(nested):
                    return nested
        except OSError as exc:
            print(f"Skip mount root {root}: {exc}")

    return None

# ── Images + metadata CSVs ───────────────────────────────────────────────────
step = time.perf_counter()
KAGGLE_INPUT = find_mount(IMAGES_SLUG, "KAGGLE_INPUT")
if not KAGGLE_INPUT:
    raise FileNotFoundError(
        f"Dataset '{IMAGES_SLUG}' not attached under {CFG['mount_search_roots']}."
    )
os.environ["KAGGLE_INPUT"] = KAGGLE_INPUT
os.environ["IMAGE_ROOT"]   = KAGGLE_INPUT
print(f"KAGGLE_INPUT (images + CSVs): {KAGGLE_INPUT} ({elapsed(step)})")

# ── Reports dataset ───────────────────────────────────────────────────────────
step = time.perf_counter()
REPORTS_ROOT = find_mount(REPORTS_SLUG, "REPORTS_ROOT")
if not REPORTS_ROOT:
    raise FileNotFoundError(
        f"Dataset '{REPORTS_SLUG}' not attached under {CFG['mount_search_roots']}."
    )
os.environ["REPORTS_ROOT"] = REPORTS_ROOT
print(f"REPORTS_ROOT:                 {REPORTS_ROOT} ({elapsed(step)})")

# ── Verify metadata CSVs ─────────────────────────────────────────────────────
step = time.perf_counter()
for fname in REQUIRED_CSVS:
    path = os.path.join(KAGGLE_INPUT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {path}")
print(f"All metadata CSVs present. ({elapsed(step)})")

# ── Preprocessed splits dataset ──────────────────────────────────────────────
step = time.perf_counter()
PROCESSED_ROOT = find_mount(PROCESSED_SLUG, "PROCESSED_ROOT")
if not PROCESSED_ROOT:
    raise FileNotFoundError(
        f"Dataset '{PROCESSED_SLUG}' not attached under {CFG['mount_search_roots']}."
    )
os.environ["PROCESSED_ROOT"] = PROCESSED_ROOT
print(f"PROCESSED_ROOT:               {PROCESSED_ROOT} ({elapsed(step)})")
for fname in PROCESSED_REQUIRED:
    p = os.path.join(PROCESSED_ROOT, fname)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")
print("All preprocessed CSVs present.")

# ── Load pre-built CSV from dataset ──────────────────────────────────────────
CSV_IN_DATASET = os.path.join(REPORTS_ROOT, CLEANED_CSV)

if not os.path.exists(CSV_IN_DATASET):
    raise FileNotFoundError(
        f"{CLEANED_CSV} not found at {CSV_IN_DATASET}.\n"
        "Run generate_mimic_cxr_cleaned.ipynb once to build it, "
        "then upload to the '{REPORTS_SLUG}' Kaggle dataset."
    )

os.makedirs(os.path.dirname(REPORTS_LOCAL), exist_ok=True)
step = time.perf_counter()
if os.path.exists(REPORTS_LOCAL) and os.path.getsize(REPORTS_LOCAL) == os.path.getsize(CSV_IN_DATASET):
    print(f"Reports CSV already present: {REPORTS_LOCAL} ({elapsed(step)})")
else:
    shutil.copy2(CSV_IN_DATASET, REPORTS_LOCAL)
    print(f"Copied reports CSV to {REPORTS_LOCAL} ({elapsed(step)})")
os.environ["REPORTS_CSV"] = REPORTS_LOCAL

import pandas as pd
step = time.perf_counter()
df = pd.read_csv(REPORTS_LOCAL)
print(f"Loaded {REPORTS_LOCAL}: {len(df)} rows ({elapsed(step)})")
print(f"\nSample:")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())


KAGGLE_INPUT (images + CSVs): /kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite
REPORTS_ROOT:                 /kaggle/input/datasets/phuong20052/mimic-cxr-reported
All metadata CSVs present.
Loaded /kaggle/working/mimic_cxr_cleaned.csv: 35162 rows

Sample:
                Img_Folder                                      Img_Filename
0  p10/p10000032/s50414267  02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg
1  p10/p10000032/s50414267  174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962.jpg
2  p10/p10000032/s53189527  2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab.jpg


In [ ]:
# Cell 4b - Cached Kaggle paths
# Run this cell instead of Cell 4 when the Kaggle dataset mounts are unchanged.
import os
import shutil
import pandas as pd

KAGGLE_INPUT   = "/kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite"
IMAGE_ROOT     = KAGGLE_INPUT
REPORTS_ROOT   = "/kaggle/input/datasets/phuong20052/mimic-cxr-reported"
PROCESSED_ROOT = "/kaggle/input/datasets/phuong20052/mimic-cxr-p10-processed"

REQUIRED_CSVS = [
    "mimic-cxr-2.0.0-split.csv",
    "mimic-cxr-2.0.0-chexpert.csv",
    "mimic-cxr-2.0.0-metadata.csv",
]
PROCESSED_REQUIRED = ["train.csv", "val.csv", "test.csv"]
CLEANED_CSV = "mimic_cxr_cleaned.csv"
REPORTS_LOCAL = "/kaggle/working/mimic_cxr_cleaned.csv"

SPLIT_CSV = os.path.join(KAGGLE_INPUT, "mimic-cxr-2.0.0-split.csv")
CHEXPERT_CSV = os.path.join(KAGGLE_INPUT, "mimic-cxr-2.0.0-chexpert.csv")
METADATA_CSV = os.path.join(KAGGLE_INPUT, "mimic-cxr-2.0.0-metadata.csv")
PROCESSED_TRAIN_CSV = os.path.join(PROCESSED_ROOT, "train.csv")
PROCESSED_VAL_CSV = os.path.join(PROCESSED_ROOT, "val.csv")
PROCESSED_TEST_CSV = os.path.join(PROCESSED_ROOT, "test.csv")

for name, path in {
    "KAGGLE_INPUT": KAGGLE_INPUT,
    "REPORTS_ROOT": REPORTS_ROOT,
    "PROCESSED_ROOT": PROCESSED_ROOT,
}.items():
    if not os.path.isdir(path):
        raise FileNotFoundError(f"{name} not found: {path}")

for fname in REQUIRED_CSVS:
    path = os.path.join(KAGGLE_INPUT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing metadata CSV: {path}")
print("All metadata CSVs present.")

for fname in PROCESSED_REQUIRED:
    path = os.path.join(PROCESSED_ROOT, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing preprocessed CSV: {path}")
print("All preprocessed CSVs present.")

CSV_IN_DATASET = os.path.join(REPORTS_ROOT, CLEANED_CSV)
if not os.path.exists(CSV_IN_DATASET):
    raise FileNotFoundError(f"{CLEANED_CSV} not found at {CSV_IN_DATASET}.")

os.makedirs(os.path.dirname(REPORTS_LOCAL), exist_ok=True)
if not os.path.exists(REPORTS_LOCAL) or os.path.getsize(REPORTS_LOCAL) != os.path.getsize(CSV_IN_DATASET):
    shutil.copy2(CSV_IN_DATASET, REPORTS_LOCAL)

os.environ["KAGGLE_INPUT"] = KAGGLE_INPUT
os.environ["IMAGE_ROOT"] = IMAGE_ROOT
os.environ["REPORTS_ROOT"] = REPORTS_ROOT
os.environ["PROCESSED_ROOT"] = PROCESSED_ROOT
os.environ["REPORTS_CSV"] = REPORTS_LOCAL
os.environ["SPLIT_CSV"] = SPLIT_CSV
os.environ["CHEXPERT_CSV"] = CHEXPERT_CSV
os.environ["METADATA_CSV"] = METADATA_CSV
os.environ["PROCESSED_TRAIN_CSV"] = PROCESSED_TRAIN_CSV
os.environ["PROCESSED_VAL_CSV"] = PROCESSED_VAL_CSV
os.environ["PROCESSED_TEST_CSV"] = PROCESSED_TEST_CSV

print(f"KAGGLE_INPUT (images + CSVs): {KAGGLE_INPUT}")
print(f"REPORTS_ROOT:                 {REPORTS_ROOT}")
print(f"PROCESSED_ROOT:               {PROCESSED_ROOT}")

df = pd.read_csv(REPORTS_LOCAL)
print(f"Loaded {REPORTS_LOCAL}: {len(df)} rows")
print("\nSample:")
print(df[["Img_Folder", "Img_Filename"]].head(3).to_string())


## Cell 5 — Write `configs/env_config.yaml` with Kaggle Paths

In [ ]:
import os
import subprocess

result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True,
)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

KAGGLE_INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input/mimic-cxr-jpg-lite")
IMAGE_ROOT = os.environ.get("IMAGE_ROOT", KAGGLE_INPUT)
REPORTS_CSV = os.environ.get("REPORTS_CSV", "/kaggle/working/mimic_cxr_cleaned.csv")
PROCESSED_ROOT = os.environ.get("PROCESSED_ROOT", "/kaggle/input/mimic-cxr-p10-processed")
GCS_BUCKET = os.environ.get("GCS_BUCKET", "gs://meta-cxr-checkpoint")
GCS_PROJECT = os.environ.get("GCS_PROJECT", "mimic-cxr-jpg-491409")

env_config_content = f"""paths:
  data_root: "{KAGGLE_INPUT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{KAGGLE_INPUT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "{REPORTS_CSV}"
  chexpert_csv: "{KAGGLE_INPUT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{KAGGLE_INPUT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "/kaggle/working/checkpoints"
  gcs_bucket: "{GCS_BUCKET}"
  gcs_project: "{GCS_PROJECT}"

wandb:
  entity: "phuongnm150505-uit"
  project: "meta-cxr-encoder-comparison"

java:
  home: "{java_home}"
  path: "{java_path}"
"""

os.makedirs("configs", exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    f.write(env_config_content)

print("Written configs/env_config.yaml:")
print(env_config_content)


## Cell 6 - Launch 2-GPU DDP Training with VM-L4-Matched Hyperparameters

This cell keeps Kaggle on 2x T4 but aligns the training setup with the GCP L4 VM comparison run:

- `max_epoch = 15`
- `init_lr/init_lr_q/init_lr_cls = 1e-4`, `min_lr = 2e-6`, `warmup_lr = 1e-6`
- effective training batch = `2 GPUs x batch_size_train 2 x accum_grad_iters 4 = 16`
- `seed = 42`
- checkpoint resume from `gs://meta-cxr-checkpoint/<run_name>/checkpoint_last.pth`
- delete the downloaded local resume `.pth` after all DDP ranks have loaded it; Cell 6 also keeps a process-exit cleanup fallback
- upload latest output and train log back to the same GCS prefix

Set `ENCODER_BIOVIL`, `ENCODER_PUBMEDCLIP`, and `ENCODER_SWIN` below before launching.


In [ ]:
import base64
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

GCS_PROJECT = "mimic-cxr-jpg-491409"
GCS_BUCKET = "meta-cxr-checkpoint"
OUTPUT_BASE = Path("/kaggle/working/output")
CHECKPOINT_BASE = Path("/kaggle/working/checkpoints")
LOG_DIR = Path("/kaggle/working/logs")

TARGET_MAX_EPOCH = 15
TRAIN_BATCH_PER_GPU = 2
EVAL_BATCH_PER_GPU = 4
ACCUM_GRAD_ITERS = 4
SEED = 42

# Encoder toggles. The run name is mapped to the same names used by meta-cxr-l4.
ENCODER_BIOVIL = True
ENCODER_PUBMEDCLIP = True
ENCODER_SWIN = False

RESUME_FROM_GCS = True
GCS_RESUME_FILENAME = "checkpoint_last.pth"
GCS_UPLOAD_AFTER_TRAIN = True
REQUIRE_GCS_CREDENTIALS = True
DELETE_LOCAL_RESUME_AFTER_LOAD = True

EXTRA_CFG_OPTIONS = []

RUN_NAME_BY_ENCODERS = {
    (True, False, False): "01_biovil_only",
    (False, True, False): "02_pubmedclip_only",
    (False, False, True): "03_swin_only",
    (True, True, False): "04_biovil_pubmedclip",
    (True, False, True): "05_biovil_swin",
    (False, True, True): "06_pubmedclip_swin",
    (True, True, True): "07_all_three",
}
RUN_NAME = RUN_NAME_BY_ENCODERS[(ENCODER_BIOVIL, ENCODER_PUBMEDCLIP, ENCODER_SWIN)]
GCS_PREFIX = RUN_NAME

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_BASE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


def _bool_option(value):
    return str(bool(value)).lower()


def _get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def _load_service_account_info():
    raw = _get_secret("GCS_SERVICE_ACCOUNT")
    if not raw:
        raw = _get_secret("GCP_SERVICE_ACCOUNT_JSON")
    if not raw:
        raw = _get_secret("GCP_SERVICE_ACCOUNT_B64")
    if not raw:
        return None

    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        decoded = base64.b64decode(raw).decode("utf-8")
        return json.loads(decoded)


def build_storage_client(required=True):
    from google.cloud import storage
    from google.oauth2 import service_account

    info = _load_service_account_info()
    if info:
        credentials = service_account.Credentials.from_service_account_info(info)
        return storage.Client(project=GCS_PROJECT, credentials=credentials)

    adc_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if adc_path and os.path.exists(adc_path):
        return storage.Client(project=GCS_PROJECT)

    if required:
        raise RuntimeError(
            "GCS credentials not found. Add Kaggle Secret GCS_SERVICE_ACCOUNT "
            "with a service-account JSON key or base64-encoded JSON. "
            f"The service account needs write access to gs://{GCS_BUCKET}."
        )
    return None


def find_latest_gcs_checkpoint(client, bucket_name, prefix, filename):
    candidates = []
    for blob in client.list_blobs(bucket_name, prefix=f"{prefix}/"):
        if blob.name == f"{prefix}/{filename}" or blob.name.endswith(f"/{filename}"):
            candidates.append(blob)
    if not candidates:
        return None
    return sorted(candidates, key=lambda b: ((b.updated.timestamp() if b.updated else 0), b.name))[-1]


def download_resume_checkpoint(client):
    if not RESUME_FROM_GCS:
        print("GCS resume disabled - starting from scratch.")
        return None

    blob = find_latest_gcs_checkpoint(client, GCS_BUCKET, GCS_PREFIX, GCS_RESUME_FILENAME)
    if blob is None:
        print(f"No {GCS_RESUME_FILENAME} found under gs://{GCS_BUCKET}/{GCS_PREFIX}/ - fresh training.")
        return None

    local_path = CHECKPOINT_BASE / RUN_NAME / GCS_RESUME_FILENAME
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(str(local_path))
    print(f"Downloaded resume checkpoint: gs://{GCS_BUCKET}/{blob.name} -> {local_path}")

    ckpt_meta = torch.load(str(local_path), map_location="cpu")
    last_epoch = int(ckpt_meta.get("epoch", -1))
    print(f"  last completed epoch = {last_epoch}; target max_epoch = {TARGET_MAX_EPOCH}")
    del ckpt_meta
    return str(local_path)


def delete_local_resume_checkpoint(path, reason):
    if not path or not DELETE_LOCAL_RESUME_AFTER_LOAD:
        return False

    p = Path(path)
    try:
        if not p.exists():
            return False
        size_mb = p.stat().st_size / (1024 ** 2)
        p.unlink()
        print(f"Deleted local resume checkpoint after {reason}: {p} ({size_mb:.1f} MB freed)")

        parent = p.parent
        if parent.exists() and parent != CHECKPOINT_BASE:
            try:
                next(parent.iterdir())
            except StopIteration:
                parent.rmdir()
        return True
    except Exception as exc:
        print(f"WARN: Could not delete local resume checkpoint {p}: {exc}")
        return False


def line_indicates_resume_loaded(line, resume_path):
    if not resume_path:
        return False
    return "Resume checkpoint from" in line and str(resume_path) in line


def ensure_runner_delete_resume_patch():
    runner_path = Path("/kaggle/working/META-CXR/model/lavis/runners/runner_base.py")
    text = runner_path.read_text(encoding="utf-8")
    changed = False

    old_call = "        # resume from checkpoint if specified\n        if not self.evaluate_only and self.resume_ckpt_path is not None:\n            self._load_checkpoint(self.resume_ckpt_path)\n            # Restore best-tracking from the checkpoint so we don't overwrite\n"
    new_call = '        # resume from checkpoint if specified\n        if not self.evaluate_only and self.resume_ckpt_path is not None:\n            self._load_checkpoint(self.resume_ckpt_path)\n            if self.config.run_cfg.get("delete_resume_ckpt_after_load", False):\n                self._delete_local_resume_checkpoint_after_load(self.resume_ckpt_path)\n            # Restore best-tracking from the checkpoint so we don\'t overwrite\n'
    if "delete_resume_ckpt_after_load" not in text:
        if old_call not in text:
            raise RuntimeError("Could not patch runner_base.py resume cleanup hook; expected resume block not found.")
        text = text.replace(old_call, new_call, 1)
        changed = True

    if "def _delete_local_resume_checkpoint_after_load" not in text:
        method = '\n    def _delete_local_resume_checkpoint_after_load(self, checkpoint_path):\n        if not checkpoint_path or is_url(checkpoint_path) or not os.path.isfile(checkpoint_path):\n            return\n\n        if dist.is_available() and dist.is_initialized():\n            dist.barrier()\n\n        if is_main_process():\n            try:\n                size_mb = os.path.getsize(checkpoint_path) / (1024 ** 2)\n                os.remove(checkpoint_path)\n                logging.info(\n                    "Deleted local resume checkpoint after load: %s (%.1f MB freed)",\n                    checkpoint_path,\n                    size_mb,\n                )\n\n                parent = Path(checkpoint_path).parent\n                if parent.exists():\n                    try:\n                        next(parent.iterdir())\n                    except StopIteration:\n                        parent.rmdir()\n            except OSError as exc:\n                logging.warning("Could not delete local resume checkpoint %s: %s", checkpoint_path, exc)\n\n        if dist.is_available() and dist.is_initialized():\n            dist.barrier()\n'
        marker = "\n    @main_process\n    def log_stats"
        if marker not in text:
            raise RuntimeError("Could not patch runner_base.py resume cleanup method; log_stats marker not found.")
        text = text.replace(marker, method + marker, 1)
        changed = True

    if changed:
        runner_path.write_text(text, encoding="utf-8")
        print(f"Patched {runner_path} to delete local resume checkpoints after all ranks load them.")
    else:
        print("runner_base.py already supports local resume checkpoint cleanup.")


def latest_output_dir(base_dir, run_name):
    candidates = []
    exact = base_dir / run_name
    if exact.exists():
        candidates.append(exact)
    candidates.extend(p for p in base_dir.glob(f"{run_name}_*") if p.is_dir())
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]


def upload_dir_to_gcs(client, local_dir, bucket_name, prefix):
    bucket = client.bucket(bucket_name)
    local_dir = Path(local_dir)
    uploaded = 0
    for path in local_dir.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(local_dir).as_posix()
        blob_name = f"{prefix.rstrip('/')}/{rel}"
        bucket.blob(blob_name).upload_from_filename(str(path))
        uploaded += 1
    print(f"Uploaded {uploaded} file(s) from {local_dir} to gs://{bucket_name}/{prefix}/")


def upload_training_artifacts(client, returncode):
    if not GCS_UPLOAD_AFTER_TRAIN:
        return

    out_dir = latest_output_dir(OUTPUT_BASE, RUN_NAME)
    if out_dir is None:
        print(f"No output directory found for {RUN_NAME}; skipping GCS output upload.")
    else:
        upload_dir_to_gcs(client, out_dir, GCS_BUCKET, GCS_PREFIX)

    log_path = LOG_DIR / f"{RUN_NAME}.log"
    if log_path.exists():
        bucket = client.bucket(GCS_BUCKET)
        bucket.blob(f"{GCS_PREFIX}/train.log").upload_from_filename(str(log_path))
        print(f"Uploaded train log to gs://{GCS_BUCKET}/{GCS_PREFIX}/train.log")

    manifest = {
        "run_name": RUN_NAME,
        "returncode": returncode,
        "target_max_epoch": TARGET_MAX_EPOCH,
        "effective_batch": TRAIN_BATCH_PER_GPU * 2 * ACCUM_GRAD_ITERS,
        "gcs_prefix": f"gs://{GCS_BUCKET}/{GCS_PREFIX}/",
    }
    manifest_path = LOG_DIR / f"{RUN_NAME}_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    client.bucket(GCS_BUCKET).blob(f"{GCS_PREFIX}/manifest.json").upload_from_filename(str(manifest_path))
    print(f"Uploaded manifest to gs://{GCS_BUCKET}/{GCS_PREFIX}/manifest.json")


ensure_runner_delete_resume_patch()

storage_client = build_storage_client(required=REQUIRE_GCS_CREDENTIALS)
resume_path = download_resume_checkpoint(storage_client) if storage_client else None

cfg_options = [
    f"model.encoders.biovil={_bool_option(ENCODER_BIOVIL)}",
    f"model.encoders.pubmedclip={_bool_option(ENCODER_PUBMEDCLIP)}",
    f"model.encoders.swin={_bool_option(ENCODER_SWIN)}",
    f"run.run_name={RUN_NAME}",
    "run.project_name=meta-cxr-encoder-comparison",
    "run.wandb_entity=phuongnm150505-uit",
    f"run.max_epoch={TARGET_MAX_EPOCH}",
    f"run.batch_size_train={TRAIN_BATCH_PER_GPU}",
    f"run.batch_size_eval={EVAL_BATCH_PER_GPU}",
    f"run.accum_grad_iters={ACCUM_GRAD_ITERS}",
    "run.init_lr=1e-4",
    "run.init_lr_q=1e-4",
    "run.init_lr_cls=1e-4",
    "run.min_lr=2e-6",
    "run.warmup_lr=1e-6",
    "run.save_freq=5",
    "run.early_stop_patience=5",
    "run.early_stop_min_delta=1e-4",
    f"run.seed={SEED}",
    "run.delete_resume_ckpt_after_load=true",
    "run.output_dir=/kaggle/working/output",
]
if resume_path:
    cfg_options.append(f"run.resume_ckpt_path={resume_path}")
cfg_options.extend(EXTRA_CFG_OPTIONS)

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    "--master_port=12355",
    "-m", "pretraining.train",
    "--cfg-path", "pretraining/configs/mimic_cxr_2gpu.yaml",
    "--options",
] + cfg_options

print("Encoder config:")
print(f"  run_name   : {RUN_NAME}")
print(f"  BioViL     : {ENCODER_BIOVIL}")
print(f"  PubMedCLIP : {ENCODER_PUBMEDCLIP}")
print(f"  Swin       : {ENCODER_SWIN}")
print("Training config:")
print(f"  max_epoch       : {TARGET_MAX_EPOCH}")
print(f"  LR              : 1e-4")
print(f"  effective batch : {TRAIN_BATCH_PER_GPU} x 2 GPUs x {ACCUM_GRAD_ITERS} = {TRAIN_BATCH_PER_GPU * 2 * ACCUM_GRAD_ITERS}")
print(f"  seed            : {SEED}")
print(f"  GCS prefix      : gs://{GCS_BUCKET}/{GCS_PREFIX}/")
print("Launch command:")
print(" ".join(cmd))
print("\n" + "=" * 60 + "\n")

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working/META-CXR"
env["GCS_BUCKET"] = f"gs://{GCS_BUCKET}"
env["GCS_PROJECT"] = GCS_PROJECT

log_path = LOG_DIR / f"{RUN_NAME}.log"
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd="/kaggle/working/META-CXR",
    env=env,
)

with open(log_path, "w", encoding="utf-8") as log_file:
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)

process.wait()
if resume_path:
    delete_local_resume_checkpoint(resume_path, "process exit cleanup")
print("\n" + "=" * 60)
print(f"Training finished with exit code: {process.returncode}")

try:
    upload_training_artifacts(storage_client, process.returncode)
except Exception as exc:
    print(f"GCS upload failed: {exc}")
    if process.returncode == 0:
        raise

if process.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {process.returncode}")


## Cell 7 - Optional Resume from `checkpoint_best.pth` on GCS

Cell 6 resumes from `checkpoint_last.pth` by default. To resume from the best checkpoint instead, set this line in Cell 6 before running it:

```python
GCS_RESUME_FILENAME = "checkpoint_best.pth"
```

The checkpoint will be downloaded from `gs://meta-cxr-checkpoint/<run_name>/checkpoint_best.pth`, passed to `pretraining.train` through `run.resume_ckpt_path`, and deleted from local Kaggle disk after all DDP ranks have loaded it.


In [ ]:
# Optional best-checkpoint resume is configured in Cell 6.
# Set GCS_RESUME_FILENAME = "checkpoint_best.pth" there, then run Cell 6.
# The downloaded local .pth file is deleted after all DDP ranks load it.
print("To resume from best checkpoint, set GCS_RESUME_FILENAME='checkpoint_best.pth' in Cell 6 and rerun Cell 6.")


## Cell 8 - Upload Latest Checkpoints to Google Cloud Storage

Run this cell if you want to manually upload the latest local output again. It uploads the newest `/kaggle/working/output/<run_name>*` directory to:

```text
gs://meta-cxr-checkpoint/<run_name>/
```

This replaces the old Kaggle Dataset checkpoint push.


In [ ]:
import base64
import json
import os
from pathlib import Path

GCS_PROJECT = "mimic-cxr-jpg-491409"
GCS_BUCKET = "meta-cxr-checkpoint"
OUTPUT_BASE = Path("/kaggle/working/output")
LOG_DIR = Path("/kaggle/working/logs")
RUN_NAME = globals().get("RUN_NAME", "04_biovil_pubmedclip")
GCS_PREFIX = RUN_NAME


def _get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def _load_service_account_info():
    raw = _get_secret("GCS_SERVICE_ACCOUNT") or _get_secret("GCP_SERVICE_ACCOUNT_JSON") or _get_secret("GCP_SERVICE_ACCOUNT_B64")
    if not raw:
        raise RuntimeError(
            "GCS credentials not found. Add Kaggle Secret GCS_SERVICE_ACCOUNT."
        )
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return json.loads(base64.b64decode(raw).decode("utf-8"))


def build_storage_client():
    from google.cloud import storage
    from google.oauth2 import service_account

    credentials = service_account.Credentials.from_service_account_info(_load_service_account_info())
    return storage.Client(project=GCS_PROJECT, credentials=credentials)


def latest_output_dir(base_dir, run_name):
    candidates = []
    exact = base_dir / run_name
    if exact.exists():
        candidates.append(exact)
    candidates.extend(p for p in base_dir.glob(f"{run_name}_*") if p.is_dir())
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]


def upload_dir_to_gcs(client, local_dir, bucket_name, prefix):
    bucket = client.bucket(bucket_name)
    local_dir = Path(local_dir)
    uploaded = 0
    for path in local_dir.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(local_dir).as_posix()
        bucket.blob(f"{prefix.rstrip('/')}/{rel}").upload_from_filename(str(path))
        uploaded += 1
    print(f"Uploaded {uploaded} file(s) from {local_dir} to gs://{bucket_name}/{prefix}/")


client = build_storage_client()
out_dir = latest_output_dir(OUTPUT_BASE, RUN_NAME)
if out_dir is None:
    raise FileNotFoundError(f"No output directory found for run {RUN_NAME} under {OUTPUT_BASE}")

upload_dir_to_gcs(client, out_dir, GCS_BUCKET, GCS_PREFIX)

log_path = LOG_DIR / f"{RUN_NAME}.log"
if log_path.exists():
    client.bucket(GCS_BUCKET).blob(f"{GCS_PREFIX}/train.log").upload_from_filename(str(log_path))
    print(f"Uploaded train log to gs://{GCS_BUCKET}/{GCS_PREFIX}/train.log")

manifest = {
    "run_name": RUN_NAME,
    "gcs_prefix": f"gs://{GCS_BUCKET}/{GCS_PREFIX}/",
    "local_output_dir": str(out_dir),
}
manifest_path = LOG_DIR / f"{RUN_NAME}_manual_upload_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
client.bucket(GCS_BUCKET).blob(f"{GCS_PREFIX}/manual_upload_manifest.json").upload_from_filename(str(manifest_path))
print(f"Done: gs://{GCS_BUCKET}/{GCS_PREFIX}/")


## Cell 9 — Compute Accuracy / Recall / F1 of `checkpoint_best.pth` on Test Split

Loads the best checkpoint, runs the MHCAC classifier over the MIMIC-CXR-JPG **test** split, and reports per-task + average accuracy, recall, and F1.

In [ ]:
import os, sys, glob
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Make sure repo is on sys.path
REPO_DIR = "/kaggle/working/META-CXR"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from model.lavis.common.config import Config
from model.lavis.common import tasks
from model.lavis.datasets.builders import load_dataset_config
from mhcac.utils import compute_metrics_for_tasks, get_task_list

# ── Locate best checkpoint ────────────────────────────────────────────────────
best_paths = sorted(glob.glob("/kaggle/working/output/**/checkpoint_best.pth", recursive=True))
if not best_paths:
    raise FileNotFoundError("checkpoint_best.pth not found under /kaggle/working/output")
BEST_CKPT = best_paths[-1]
print(f"Best checkpoint: {BEST_CKPT}")

# ── Build cfg (single-GPU, no DDP) ───────────────────────────────────────────
def bool_option(value):
    return str(bool(value)).lower()

_eval_biovil = globals().get("ENCODER_BIOVIL", True)
_eval_pubmedclip = globals().get("ENCODER_PUBMEDCLIP", True)
_eval_swin = globals().get("ENCODER_SWIN", False)

class _Args:
    cfg_path = "pretraining/configs/mimic_cxr_2gpu.yaml"
    options = [
        "run.distributed=False",
        "run.world_size=1",
        "run.evaluate=True",
        f"model.encoders.biovil={bool_option(_eval_biovil)}",
        f"model.encoders.pubmedclip={bool_option(_eval_pubmedclip)}",
        f"model.encoders.swin={bool_option(_eval_swin)}",
    ]

cfg = Config(_Args())
cfg.run_cfg.distributed = False

task = tasks.setup_task(cfg)

# ── Build test dataset ───────────────────────────────────────────────────────
# The MIMIC builder takes a split argument; we instantiate it directly for "test".
from model.lavis.common.registry import registry as _reg
builder_cls = _reg.get_builder_class("mimic_cxr")
builder = builder_cls(cfg)
builder.build_processors()
datasets_cfg = cfg.datasets_cfg.mimic_cxr
vis_processor = builder.vis_processors["eval"]
text_processor = builder.text_processors["eval"]

from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
test_ds = MIMIC_CXR_Dataset(
    vis_processor=vis_processor,
    text_processor=text_processor,
    vis_root=os.environ.get("IMAGE_ROOT", "/kaggle/input/mimic-cxr-jpg-lite"),
    split="test",
    cfg=cfg,
    truncate=None,
)
print(f"Test samples: {len(test_ds)}")

# ── Build model + load best weights ──────────────────────────────────────────
model = task.build_model(cfg)
sd = torch.load(BEST_CKPT, map_location="cpu")
missing, unexpected = model.load_state_dict(sd["model"], strict=False)
print(f"load_state_dict: {len(missing)} missing, {len(unexpected)} unexpected")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

# ── Inference loop ───────────────────────────────────────────────────────────
loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

all_logits, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(loader, desc="Test"):
        images = batch["image"].to(device)
        cls_logits, _ = model.forward_image(images)
        all_logits.append(cls_logits.detach().cpu())
        all_labels.append(batch["classification_labels"].cpu())

logits = torch.cat(all_logits, dim=0)
labels = torch.cat(all_labels, dim=0)
print(f"Aggregated logits {tuple(logits.shape)}, labels {tuple(labels.shape)}")

# ── Compute metrics ──────────────────────────────────────────────────────────
metrics = compute_metrics_for_tasks(logits, labels)

rows = []
for task_name in get_task_list():
    m = metrics[task_name]
    rows.append({
        "task":      task_name,
        "accuracy":  float(m["accuracy"]),
        "recall":    float(m["recall"]),
        "f1_score":  float(m["f1_score"]),
        "precision": float(m["precision"]),
    })
avg = metrics["average"]
rows.append({
    "task":      "AVERAGE",
    "accuracy":  float(avg["accuracy"]),
    "recall":    float(avg["recall"]),
    "f1_score":  float(avg["f1_score"]),
    "precision": float(avg["precision"]),
})

df = pd.DataFrame(rows)
pd.options.display.float_format = "{:.4f}".format
print(df.to_string(index=False))
try:
    display(df)
except NameError:
    pass